##Data ingestion

In [2]:
from google.colab import files
uploaded = files.upload()

Saving capacitiesHosp.csv to capacitiesHosp.csv
Saving pvtHosp.csv to pvtHosp.csv
Saving pvtClinicsHosp.csv to pvtClinicsHosp.csv
Saving govtHosp.csv to govtHosp.csv
Saving dispensariesHosp.csv to dispensariesHosp.csv


##Data Cleaning

In [27]:
!pip install pandas geopy

In [28]:
!ls

capacitiesHosp.csv	  dispensariesHosp.csv	pvtClinicsHosp.csv  sample_data
clean_pune_hospitals.csv  govtHosp.csv		pvtHosp.csv


In [29]:
import pandas as pd

capacities = pd.read_csv("capacitiesHosp.csv")
govt = pd.read_csv("govtHosp.csv")
pvt = pd.read_csv("pvtHosp.csv")
clinics = pd.read_csv("pvtClinicsHosp.csv")
disp = pd.read_csv("dispensariesHosp.csv")

In [30]:
print("GOVT:", govt.columns)
print("PRIVATE:", pvt.columns)
print("CLINICS:", clinics.columns)
print("DISPENSARIES:", disp.columns)
print("CAPACITY:", capacities.columns)

GOVT: Index(['City Name', 'Ward No', 'Ward Name', 'Name of healthcare facility',
       'Type of facility', 'Latitude', 'Longitude'],
      dtype='object')
PRIVATE: Index(['City Name', 'Ward No', 'Ward Name', 'Name of healthcare facility',
       'Type of facility', 'Latitude', 'Longitude'],
      dtype='object')
CLINICS: Index(['City Name', 'Ward No', 'Ward Name', 'Name of healthcare facility',
       'Type of facility', 'Latitude', 'Longitude'],
      dtype='object')
DISPENSARIES: Index(['City Name', 'Ward No', 'Ward Name', 'Name of healthcare facility',
       'Type of facility', 'Latitude', 'Longitude'],
      dtype='object')
CAPACITY: Index(['City Name', 'Zone Name', 'Ward Name', 'Zone No.', 'Ward No.',
       'Facility Name', 'Type  (Hospital / Nursing Home / Lab)',
       ' Class : (Public / Private)', 'Level (Primary / Secondary / Tertiary)',
       'Pharmacy Available : Yes/No', 'Number of Beds in Emergency Wards ',
       'Number of Beds in facility type', 'Number of Doctors 

In [31]:
govt_clean = govt.rename(columns={
    "Name of healthcare facility": "name",
    "Latitude": "lat",
    "Longitude": "long",
    "Type of facility": "facility_type"
})

govt_clean["ownership"] = "Government"

In [32]:
pvt_clean = pvt.rename(columns={
    "Name of healthcare facility": "name",
    "Latitude": "lat",
    "Longitude": "long",
    "Type of facility": "facility_type"
})

pvt_clean["ownership"] = "Private"

In [33]:
clinic_clean = clinics.rename(columns={
    "Name of healthcare facility": "name",
    "Latitude": "lat",
    "Longitude": "long",
    "Type of facility": "facility_type"
})

clinic_clean["ownership"] = "Private"

In [34]:
disp_clean = disp.rename(columns={
    "Name of healthcare facility": "name",
    "Latitude": "lat",
    "Longitude": "long",
    "Type of facility": "facility_type"
})

disp_clean["ownership"] = "Government"

In [35]:
cols = ["name", "lat", "long", "facility_type", "ownership"]

govt_clean = govt_clean[cols]
pvt_clean = pvt_clean[cols]
clinic_clean = clinic_clean[cols]
disp_clean = disp_clean[cols]

In [36]:
hospitals_all = pd.concat(
    [govt_clean, pvt_clean, clinic_clean, disp_clean],
    ignore_index=True
)

print("Total facilities (raw):", hospitals_all.shape[0])
hospitals_all.head()

Total facilities (raw): 478


,name,lat,long,facility_type,ownership
0,Late. Jayabai Nanasaheb Sutar Maternity Home,18.503652,73.807762,Government hospital(PMC),Government
1,Late. Sundarabai Ganpatrao Raut Dawakhana,18.511578,73.820137,Government hospital(PMC),Government
2,Sanjay Gandhi Maternity Home,18.571104,73.838294,Government hospital(PMC),Government
3,Late. Sahadev Eknath Nimhan Maternity Home,18.537722,73.795979,Government hospital(PMC),Government
4,Aundh Kuti Maternity Home,18.562805,73.810015,Government hospital(PMC),Government


In [37]:
hospitals_all = hospitals_all.dropna(subset=["lat", "long"])

In [38]:
hospitals_all = hospitals_all.drop_duplicates(subset=["name"])

In [39]:
capacity_clean = capacities.rename(columns={
    "Facility Name": "name",
    "Number of Beds in Emergency Wards ": "emergency_beds",
    "Ambulance Service Available": "ambulance_available"
})

In [40]:
capacity_clean = capacity_clean[
    ["name", "emergency_beds", "ambulance_available"]
]

In [41]:
hospitals_all = hospitals_all.merge(
    capacity_clean,
    on="name",
    how="left"
)

In [42]:
hospitals_all = hospitals_all.sort_values(by=["lat", "long"])

In [43]:
hospitals_all.to_csv("clean_pune_hospitals.csv", index=False)

In [44]:
files.download("clean_pune_hospitals.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [45]:
df= pd.read_csv("clean_pune_hospitals.csv")

In [46]:
def generate_dataset_description(df):
    desc = []

    desc.append("DATASET: Pune Healthcare Facilities (Cleaned)\n")
    desc.append(f"Total Records (Facilities): {len(df)}")
    desc.append(f"Total Features (Columns): {df.shape[1]}\n")

    desc.append("COLUMN-WISE DESCRIPTION:\n")

    for col in df.columns:
        col_desc = {
            "column": col,
            "dtype": str(df[col].dtype),
            "missing_values": int(df[col].isna().sum()),
            "unique_values": df[col].nunique(),
            "sample_values": df[col].dropna().unique()[:3]
        }

        desc.append(
            f"""
• {col}
  - Data Type: {col_desc['dtype']}
  - Missing Values: {col_desc['missing_values']}
  - Unique Values: {col_desc['unique_values']}
  - Sample Values: {list(col_desc['sample_values'])}
"""
        )

    return "\n".join(desc)

dataset_description = generate_dataset_description(df)
print(dataset_description)

DATASET: Pune Healthcare Facilities (Cleaned)

Total Records (Facilities): 455
Total Features (Columns): 7

COLUMN-WISE DESCRIPTION:


• name
  - Data Type: object
  - Missing Values: 0
  - Unique Values: 448
  - Sample Values: ['Bharti Homeopathic Hospitalnan', 'Bharti Hospitalnan', 'Bharti Ayurved Hospitalnan']


• lat
  - Data Type: float64
  - Missing Values: 0
  - Unique Values: 448
  - Sample Values: [np.float64(18.4588916), np.float64(18.4589235), np.float64(18.4589627)]


• long
  - Data Type: float64
  - Missing Values: 0
  - Unique Values: 448
  - Sample Values: [np.float64(73.85680646), np.float64(73.85678324), np.float64(73.85625754)]


• facility_type
  - Data Type: object
  - Missing Values: 0
  - Unique Values: 4
  - Sample Values: ['private hospital', 'Government hospital(PMC)', 'private clinic']


• ownership
  - Data Type: object
  - Missing Values: 0
  - Unique Values: 2
  - Sample Values: ['Private', 'Government']


• emergency_beds
  - Data Type: object
  - Missing

In [47]:
ml_roles = {
    "IDENTIFIERS": [],
    "GEO_FEATURES": [],
    "CATEGORICAL_FEATURES": [],
    "NUMERICAL_FEATURES": [],
    "TARGET_CANDIDATES": []
}

for col in df.columns:
    if col.lower() in ["latitude", "longitude"]:
        ml_roles["GEO_FEATURES"].append(col)
    elif df[col].dtype == "object":
        ml_roles["CATEGORICAL_FEATURES"].append(col)
    else:
        ml_roles["NUMERICAL_FEATURES"].append(col)

ml_roles

{'IDENTIFIERS': [],
 'GEO_FEATURES': [],
 'CATEGORICAL_FEATURES': ['name',
  'facility_type',
  'ownership',
  'emergency_beds',
  'ambulance_available'],
 'NUMERICAL_FEATURES': ['lat', 'long'],
 'TARGET_CANDIDATES': []}